# E-Commerce Sales & Customer Analytics

**SQL + Python analysis of a 12-month e-commerce dataset.**

This notebook takes four raw CSV exports through to a set of costed business
recommendations: model the data relationally, validate it, analyse it in SQL,
segment the customer base with RFM, and translate the findings into actions.

### Contents

| # | Section |
|---|---|
| 1 | Data loading and quality validation |
| 2 | Relational model (SQLite) |
| 3 | Business overview and revenue reconciliation |
| 4 | Sales trends and monthly performance |
| 5 | Product and category performance |
| 6 | Geographic performance |
| 7 | Customer segmentation — RFM |
| 8 | Visualisations |
| 9 | Executive dashboard |
| 10 | Insights and recommendations |

### Method notes

- **All revenue is scoped to completed orders.** Cancelled and returned orders
  are excluded from every monetary figure in this notebook.
- **Recency is measured against the dataset, not today.** RFM uses the most
  recent completed order date as the reference point, so the analysis
  reproduces identically whenever it is re-run.
- **No figure is hardcoded.** Every number in the insights and summary sections
  is computed at runtime from the dataframes above it.

### Stack

Python · pandas · SQLite (CTEs, window functions) · Matplotlib · Seaborn · Plotly

---
## 0. Setup

In [ ]:
import os
import sqlite3
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# Works both locally (repo layout) and in Colab (files uploaded to /content)
DATA_DIR = "data/raw" if os.path.isdir("data/raw") else "."

print("pandas", pd.__version__)
print("data directory:", os.path.abspath(DATA_DIR))

---
## 1. Data loading and quality validation

Four tables, loaded and profiled before anything is analysed. The checks below
are the ones worth running before trusting any downstream number: shape, nulls,
duplicates, primary-key uniqueness, and referential integrity across the joins
this analysis depends on.

In [ ]:
products    = pd.read_csv(f"{DATA_DIR}/products.csv")
customers   = pd.read_csv(f"{DATA_DIR}/customers.csv")
orders      = pd.read_csv(f"{DATA_DIR}/orders.csv")
order_items = pd.read_csv(f"{DATA_DIR}/order_items.csv")

tables = {
    "products":    products,
    "customers":   customers,
    "orders":      orders,
    "order_items": order_items,
}

profile = pd.DataFrame([
    {
        "table":      name,
        "rows":       len(df),
        "columns":    df.shape[1],
        "missing":    int(df.isna().sum().sum()),
        "duplicates": int(df.duplicated().sum()),
    }
    for name, df in tables.items()
])

profile

In [ ]:
# Primary-key uniqueness and referential integrity.
# Every one of these must be zero before the joins below can be trusted.

integrity = pd.DataFrame([
    ("Duplicate customer_id", int(customers["customer_id"].duplicated().sum())),
    ("Duplicate order_id",    int(orders["order_id"].duplicated().sum())),
    ("Duplicate product_id",  int(products["product_id"].duplicated().sum())),
    ("Orders with no matching customer",
     int((~orders["customer_id"].isin(customers["customer_id"])).sum())),
    ("Order items with no matching order",
     int((~order_items["order_id"].isin(orders["order_id"])).sum())),
    ("Order items with no matching product",
     int((~order_items["product_id"].isin(products["product_id"])).sum())),
], columns=["check", "failures"])

integrity["result"] = integrity["failures"].map(lambda n: "PASS" if n == 0 else "REVIEW")
integrity

---
## 2. Relational model

The CSVs are loaded into SQLite so the analysis can be written in SQL rather
than chained pandas operations. This mirrors how the work would run against a
production warehouse, and keeps the queries portable — every one below runs
unchanged against Postgres or BigQuery with only date-function substitutions.

```
customers ──< orders ──< order_items >── products
```

In [ ]:
conn = sqlite3.connect(":memory:")

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)


def run_sql(query: str) -> pd.DataFrame:
    """Execute a query against the in-memory database and return a DataFrame."""
    return pd.read_sql_query(query, conn)


run_sql("""
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

---
## 3. Business overview and revenue reconciliation

In [ ]:
order_status = run_sql("""
    SELECT
        status,
        COUNT(*) AS order_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct_of_orders
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC;
""")

order_status

### Reconciliation: not every completed order carries revenue

A count of orders marked `Completed` and a count of orders that actually join to
`order_items` do not agree. Some completed orders have no line items attached,
so they contribute nothing to revenue.

This is worth surfacing rather than silently absorbing. Every revenue figure in
this notebook is computed over the orders that join cleanly — the gap below is
the population those figures exclude, and in a live engagement it is the first
question to take back to the client's data team.

In [ ]:
reconciliation = run_sql("""
    SELECT
        (SELECT COUNT(*)
         FROM orders
         WHERE status = 'Completed')                       AS completed_orders,

        (SELECT COUNT(DISTINCT o.order_id)
         FROM orders o
         JOIN order_items oi ON o.order_id = oi.order_id
         WHERE o.status = 'Completed')                     AS with_line_items;
""")

reconciliation["orphaned"] = (
    reconciliation["completed_orders"] - reconciliation["with_line_items"]
)
reconciliation["orphaned_pct"] = (
    reconciliation["orphaned"] / reconciliation["completed_orders"] * 100
).round(2)

reconciliation

In [ ]:
# Headline totals. Note that order and customer counts use COUNT(DISTINCT) —
# they are not additive across categories or countries the way revenue is.

totals = run_sql("""
    SELECT
        COUNT(DISTINCT o.order_id)              AS completed_orders,
        COUNT(DISTINCT o.customer_id)           AS purchasing_customers,
        SUM(oi.quantity)                        AS units_sold,
        ROUND(SUM(oi.quantity * oi.price), 2)   AS total_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed';
""").iloc[0]

total_orders    = int(totals["completed_orders"])
total_customers = int(totals["purchasing_customers"])
total_units     = int(totals["units_sold"])
total_revenue   = float(totals["total_revenue"])
average_order_value = round(total_revenue / total_orders, 2)

pd.DataFrame([
    ("Total revenue",       f"${total_revenue:,.2f}"),
    ("Completed orders",    f"{total_orders:,}"),
    ("Purchasing customers", f"{total_customers:,}"),
    ("Units sold",          f"{total_units:,}"),
    ("Average order value", f"${average_order_value:,.2f}"),
], columns=["metric", "value"])

---
## 4. Sales trends and monthly performance

Month-over-month growth is computed with a `LAG` window function over a CTE
rather than in pandas, keeping the aggregation in the database.

In [ ]:
monthly_growth = run_sql("""
    WITH monthly AS (
        SELECT
            strftime('%Y-%m', o.order_date)     AS month,
            COUNT(DISTINCT o.order_id)          AS orders,
            SUM(oi.quantity)                    AS units_sold,
            SUM(oi.quantity * oi.price)         AS revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'Completed'
        GROUP BY month
    )
    SELECT
        month,
        orders,
        units_sold,
        ROUND(revenue, 2) AS revenue,
        ROUND(
            (revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
            / LAG(revenue) OVER (ORDER BY month),
            2
        ) AS revenue_growth_pct
    FROM monthly
    ORDER BY month;
""")

monthly_growth

---
## 5. Product and category performance

In [ ]:
product_performance = run_sql("""
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        COUNT(DISTINCT oi.order_id)             AS orders,
        SUM(oi.quantity)                        AS units_sold,
        ROUND(SUM(oi.quantity * oi.price), 2)   AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders   o ON oi.order_id   = o.order_id
    WHERE o.status = 'Completed'
    GROUP BY p.product_id, p.product_name, p.category
    ORDER BY revenue DESC;
""")

product_performance.head(10)

In [ ]:
category_performance = run_sql("""
    SELECT
        p.category,
        COUNT(DISTINCT p.product_id)            AS products,
        COUNT(DISTINCT oi.order_id)             AS orders,
        SUM(oi.quantity)                        AS units_sold,
        ROUND(SUM(oi.quantity * oi.price), 2)   AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders   o ON oi.order_id   = o.order_id
    WHERE o.status = 'Completed'
    GROUP BY p.category
    ORDER BY revenue DESC;
""")

category_performance["revenue_share_pct"] = (
    category_performance["revenue"] / category_performance["revenue"].sum() * 100
).round(2)

category_performance

---
## 6. Geographic performance

In [ ]:
country_performance = run_sql("""
    SELECT
        c.country,
        COUNT(DISTINCT c.customer_id)           AS customers,
        COUNT(DISTINCT o.order_id)              AS orders,
        SUM(oi.quantity)                        AS units_sold,
        ROUND(SUM(oi.quantity * oi.price), 2)   AS revenue
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY c.country
    ORDER BY revenue DESC;
""")

country_performance["revenue_per_customer"] = (
    country_performance["revenue"] / country_performance["customers"]
).round(2)

country_performance

In [ ]:
country_category = run_sql("""
    SELECT
        c.country,
        p.category,
        ROUND(SUM(oi.quantity * oi.price), 2) AS revenue
    FROM order_items oi
    JOIN orders    o ON oi.order_id    = o.order_id
    JOIN customers c ON o.customer_id  = c.customer_id
    JOIN products  p ON oi.product_id  = p.product_id
    WHERE o.status = 'Completed'
    GROUP BY c.country, p.category;
""")

country_category_pivot = (
    country_category
    .pivot(index="country", columns="category", values="revenue")
    .fillna(0)
)

country_category_pivot

---
## 7. Customer segmentation — RFM

Customers are scored 1–5 on three dimensions and grouped into named segments:

- **Recency** — days since their last completed order
- **Frequency** — number of completed orders
- **Monetary** — total spend

**Scoring convention: 5 is always best on all three dimensions.**

This matters more than it sounds. Frequency and monetary scores rise with the
underlying value, so a plain quintile works. Recency runs the other way — a
*low* day-count means a *recent* buyer, which deserves a *high* score — so the
recency tile has to be inverted. Getting that backwards silently relabels your
best customers as your worst ones and inverts every recommendation that
follows, so the assertion below verifies it rather than assuming it.

Recency is measured relative to the latest completed order **in the dataset**,
not today's date, so re-running this next year gives the same answer.

In [ ]:
rfm = run_sql("""
    WITH customer_rfm AS (
        SELECT
            o.customer_id,
            MAX(o.order_date)               AS last_purchase_date,
            COUNT(DISTINCT o.order_id)      AS frequency,
            SUM(oi.quantity * oi.price)     AS monetary_value
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'Completed'
        GROUP BY o.customer_id
    ),

    reference_date AS (
        SELECT MAX(order_date) AS max_date
        FROM orders
        WHERE status = 'Completed'
    ),

    rfm_metrics AS (
        SELECT
            r.customer_id,
            CAST(julianday(d.max_date) - julianday(r.last_purchase_date)
                 AS INTEGER)                AS recency,
            r.frequency,
            r.monetary_value
        FROM customer_rfm r
        CROSS JOIN reference_date d
    ),

    rfm_scores AS (
        SELECT
            customer_id,
            recency,
            frequency,
            ROUND(monetary_value, 2) AS monetary_value,

            -- inverted: fewer days since purchase must score higher
            6 - NTILE(5) OVER (ORDER BY recency)        AS recency_score,
            NTILE(5)     OVER (ORDER BY frequency)      AS frequency_score,
            NTILE(5)     OVER (ORDER BY monetary_value) AS monetary_score
        FROM rfm_metrics
    )

    SELECT
        *,
        recency_score + frequency_score + monetary_score AS rfm_score
    FROM rfm_scores
    ORDER BY rfm_score DESC, monetary_value DESC;
""")

rfm.head(10)

In [ ]:
# Guardrail: customers scoring 5 on recency must be the ones who bought
# most recently. If this ever fires, the tile inversion has been lost.

check = rfm.groupby("recency_score")["recency"].mean()

assert check.loc[5] < check.loc[1], (
    "Recency scoring is inverted — score 5 must mean a recent buyer."
)

print("Recency direction verified.\n")
check.round(1).rename("avg_days_since_last_purchase").to_frame()

In [ ]:
def segment_customer(row):
    r, f, m = row["recency_score"], row["frequency_score"], row["monetary_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"            # recent, frequent, high value
    if r >= 3 and f >= 4 and m >= 4:
        return "Loyal Customers"      # slightly less recent, still valuable
    if r >= 4 and f >= 2:
        return "Potential Loyalists"  # recent, building a habit
    if r <= 2 and f >= 4 and m >= 4:
        return "At Risk"              # was valuable, has gone quiet
    if r <= 2 and f <= 2:
        return "Lost Customers"
    if r >= 3 and f <= 2:
        return "New Customers"
    return "Needs Attention"


rfm["segment"] = rfm.apply(segment_customer, axis=1)

rfm_segments = (
    rfm.groupby("segment")
       .agg(
           customers=("customer_id", "count"),
           revenue=("monetary_value", "sum"),
           avg_revenue=("monetary_value", "mean"),
           avg_recency_days=("recency", "mean"),
           avg_frequency=("frequency", "mean"),
       )
       .sort_values("revenue", ascending=False)
       .round(2)
       .reset_index()
)

rfm_segments["revenue_share_pct"] = (
    rfm_segments["revenue"] / rfm_segments["revenue"].sum() * 100
).round(2)

rfm_segments

---
## 8. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=rfm_segments, x="segment", y="customers",
            hue="segment", palette="Blues_d", legend=False, ax=axes[0])
axes[0].set_title("Customers by RFM segment")
axes[0].set_xlabel("")
axes[0].set_ylabel("Customers")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=rfm_segments, x="segment", y="revenue",
            hue="segment", palette="Oranges_d", legend=False, ax=axes[1])
axes[1].set_title("Revenue by RFM segment")
axes[1].set_xlabel("")
axes[1].set_ylabel("Revenue")
axes[1].tick_params(axis="x", rotation=45)

for label in axes[0].get_xticklabels() + axes[1].get_xticklabels():
    label.set_ha("right")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=category_performance, x="category", y="revenue",
            hue="category", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Revenue by product category")
axes[0].set_xlabel("")
axes[0].set_ylabel("Revenue")

axes[1].pie(category_performance["revenue_share_pct"],
            labels=category_performance["category"],
            autopct="%1.1f%%", startangle=90,
            colors=sns.color_palette("viridis", len(category_performance)))
axes[1].set_title("Revenue share by category")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(monthly_growth["month"], monthly_growth["revenue"],
        marker="o", linewidth=2, color="#2563EB")
ax.axhline(monthly_growth["revenue"].mean(), linestyle="--",
           color="#94A3B8", linewidth=1, label="Monthly average")

ax.set_title("Monthly revenue trend")
ax.set_xlabel("")
ax.set_ylabel("Revenue")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(country_category_pivot, annot=True, fmt=",.0f",
            cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Revenue"})

plt.title("Revenue by country and product category")
plt.xlabel("Category")
plt.ylabel("")
plt.tight_layout()
plt.show()

---
## 9. Executive dashboard

A single Plotly figure combining the headline KPIs, segment and category
breakdowns, the monthly trend, and the geographic heatmap.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TEXT, MUTED = "#0F172A", "#64748B"
BLUE, ORANGE, GREEN = "#2563EB", "#EA580C", "#16A34A"

monthly_plot = monthly_growth.copy()
monthly_plot["label"] = pd.to_datetime(monthly_plot["month"]).dt.strftime("%b %Y")

fig = make_subplots(
    rows=4, cols=2,
    row_heights=[0.14, 0.30, 0.30, 0.26],
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "bar"},       {"type": "bar"}],
        [{"type": "scatter"},   {"type": "heatmap"}],
        [{"type": "table", "colspan": 2}, None],
    ],
    vertical_spacing=0.09,
    horizontal_spacing=0.08,
    subplot_titles=(
        "", "",
        "Revenue by customer segment", "Revenue by product category",
        "Monthly revenue trend", "Revenue by country and category",
        "Top products by revenue",
    ),
)

fig.add_trace(go.Indicator(
    mode="number", value=total_revenue,
    number={"prefix": "$", "valueformat": ",.0f",
            "font": {"size": 34, "color": TEXT}},
    title={"text": "Total revenue", "font": {"size": 13, "color": MUTED}},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=average_order_value,
    number={"prefix": "$", "valueformat": ",.2f",
            "font": {"size": 34, "color": TEXT}},
    title={"text": f"Average order value  ·  {total_orders:,} orders",
           "font": {"size": 13, "color": MUTED}},
), row=1, col=2)

fig.add_trace(go.Bar(
    x=rfm_segments["segment"], y=rfm_segments["revenue"],
    marker_color=BLUE, showlegend=False,
    hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>",
), row=2, col=1)

fig.add_trace(go.Bar(
    x=category_performance["category"], y=category_performance["revenue"],
    marker_color=GREEN, showlegend=False,
    hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>",
), row=2, col=2)

fig.add_trace(go.Scatter(
    x=monthly_plot["label"], y=monthly_plot["revenue"],
    mode="lines+markers", line={"color": ORANGE, "width": 3},
    showlegend=False, hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>",
), row=3, col=1)

fig.add_trace(go.Heatmap(
    z=country_category_pivot.values,
    x=country_category_pivot.columns.tolist(),
    y=country_category_pivot.index.tolist(),
    colorscale="YlGnBu", showscale=False,
    hovertemplate="%{y} · %{x}<br>$%{z:,.0f}<extra></extra>",
), row=3, col=2)

top_products = product_performance.head(8)
fig.add_trace(go.Table(
    header={"values": ["Product", "Category", "Units", "Revenue"],
            "fill_color": "#F1F5F9", "align": "left",
            "font": {"color": TEXT, "size": 12}},
    cells={"values": [
               top_products["product_name"],
               top_products["category"],
               top_products["units_sold"],
               top_products["revenue"].map(lambda v: f"${v:,.0f}"),
           ],
           "fill_color": "white", "align": "left",
           "font": {"color": MUTED, "size": 11}, "height": 24},
), row=4, col=1)

fig.update_layout(
    height=1250, template="plotly_white",
    title={"text": "E-Commerce Performance Dashboard",
           "font": {"size": 24, "color": TEXT}, "x": 0.5, "xanchor": "center"},
    margin={"t": 110, "b": 40, "l": 60, "r": 60},
)
fig.update_annotations(font_size=13)
fig.update_xaxes(tickangle=-40, row=2, col=1)
fig.update_xaxes(tickangle=-40, row=3, col=1)

os.makedirs("outputs", exist_ok=True)
fig.write_html("outputs/dashboard.html", include_plotlyjs="cdn")

fig.show()

---
## 10. Insights and recommendations

Every figure below is derived from the dataframes above at runtime. Nothing is
typed in by hand, so the narrative cannot drift out of sync with the data.

In [ ]:
def build_insights():
    top_segment  = rfm_segments.iloc[0]
    top_category = category_performance.iloc[0]
    weak_category = category_performance.iloc[-1]
    top_country  = country_performance.iloc[0]

    best_month   = monthly_growth.loc[monthly_growth["revenue"].idxmax()]
    worst_month  = monthly_growth.loc[monthly_growth["revenue"].idxmin()]
    best_growth  = monthly_growth.loc[monthly_growth["revenue_growth_pct"].idxmax()]

    def seg(name):
        row = rfm_segments.loc[rfm_segments["segment"] == name]
        return row.iloc[0] if len(row) else None

    champions = seg("Champions")
    at_risk   = seg("At Risk")

    out = [
        "=" * 66,
        "BUSINESS INSIGHTS",
        "=" * 66,
        "",
        "1. HEADLINE",
        f"   ${total_revenue:,.0f} revenue · {total_orders:,} completed orders · "
        f"{total_customers:,} customers",
        f"   Average order value ${average_order_value:,.2f} · {total_units:,} units sold",
        "",
        "2. CUSTOMER SEGMENTS",
        f"   Largest revenue segment: {top_segment['segment']} — "
        f"${top_segment['revenue']:,.0f} ({top_segment['revenue_share_pct']:.1f}%) "
        f"from {int(top_segment['customers'])} customers",
    ]

    if champions is not None:
        out.append(
            f"   Champions: {int(champions['customers'])} customers averaging "
            f"${champions['avg_revenue']:,.0f} each"
        )
    if at_risk is not None:
        out.append(
            f"   At Risk: {int(at_risk['customers'])} customers worth "
            f"${at_risk['revenue']:,.0f}, averaging "
            f"{at_risk['avg_recency_days']:.0f} days since last order"
        )

    out += [
        "",
        "3. PRODUCT CATEGORIES",
        f"   Strongest: {top_category['category']} — ${top_category['revenue']:,.0f} "
        f"({top_category['revenue_share_pct']:.1f}% of revenue)",
        f"   Weakest:   {weak_category['category']} — ${weak_category['revenue']:,.0f} "
        f"({weak_category['revenue_share_pct']:.1f}%)",
        "",
        "4. MONTHLY PERFORMANCE",
        f"   Best month:    {best_month['month']} at ${best_month['revenue']:,.0f}",
        f"   Weakest month: {worst_month['month']} at ${worst_month['revenue']:,.0f}",
        f"   Strongest growth: {best_growth['month']} at "
        f"{best_growth['revenue_growth_pct']:+.1f}% month-over-month",
        "",
        "5. GEOGRAPHY",
        f"   Top market: {top_country['country']} — ${top_country['revenue']:,.0f} "
        f"from {int(top_country['customers'])} customers "
        f"(${top_country['revenue_per_customer']:,.0f} per customer)",
        "",
        "=" * 66,
        "RECOMMENDED ACTIONS",
        "=" * 66,
        "",
    ]

    actions = []
    if at_risk is not None and at_risk["revenue"] > 0:
        actions.append(
            f"Run a win-back campaign aimed at the {int(at_risk['customers'])} "
            f"At Risk customers — ${at_risk['revenue']:,.0f} of proven spend "
            f"that has gone quiet, averaging "
            f"{at_risk['avg_recency_days']:.0f} days since the last order. "
            f"Purchase intent is already demonstrated here, so the cost per "
            f"recovered order should undercut new acquisition."
        )
    if champions is not None:
        actions.append(
            f"Protect the {int(champions['customers'])} Champions with early "
            f"access and loyalty benefits. Small group, "
            f"${champions['avg_revenue']:,.0f} average value — churn here is "
            f"expensive and quiet."
        )
    actions += [
        f"Lead acquisition and cross-sell with {top_category['category']}. "
        f"It carries {top_category['revenue_share_pct']:.1f}% of revenue and is "
        f"the natural anchor for bundles.",

        f"Diagnose {weak_category['category']} before investing further. At "
        f"{weak_category['revenue_share_pct']:.1f}% of revenue, establish "
        f"whether the constraint is assortment, price, or demand — the fix "
        f"differs sharply in each case.",

        f"Study what drove {best_growth['month']} "
        f"({best_growth['revenue_growth_pct']:+.1f}% MoM) and test replaying it "
        f"into {worst_month['month']}-type troughs.",

        f"Resolve the {int(reconciliation['orphaned'].iloc[0])} completed orders "
        f"with no line items "
        f"({reconciliation['orphaned_pct'].iloc[0]:.1f}% of completed orders). "
        f"Until that is explained, reported revenue carries a known gap.",
    ]

    for i, action in enumerate(actions, 1):
        out.append(f"{i}. {action}")
        out.append("")

    return "\n".join(out)


print(build_insights())

---
## Limitations

Worth stating plainly, because analysis that hides its own caveats is worth
less than analysis that names them:

- **The orphaned completed orders are unexplained.** They are excluded from
  revenue and quantified above, but the root cause sits upstream of this data.
- **RFM quintiles are relative, not absolute.** `NTILE(5)` splits *this*
  customer base into fifths, so a score of 5 means "top fifth here", not a
  threshold that holds across time periods or comparable businesses.
- **Twelve months gives one observation per calendar month.** What reads as
  seasonality cannot be separated from one-off events without a second year.
- **Revenue is gross.** No discounts, shipping, returns value, refunds, or cost
  of goods, so none of these figures are margin.
- **Segment boundaries are a judgement call.** The score cut-offs are a
  conventional starting point and should be tuned against the client's actual
  purchase cycle before driving spend.